# AWS Executive Cost Report
Builds the executive HTML report from the `Monthly_Cost_For_*.csv` extracts produced by **FinOps.ipynb**.

* **AWS Services** (tracked against the monthly budget) are separated from **Marketplace** and **Tax**.
* All logic lives in `finops_exec_report.py`, so the same report can be run from the terminal:
  `python finops_exec_report.py --month 2026-09 --budget 550000`

In [1]:
import importlib
import pandas as pd
import finops_exec_report as fr
importlib.reload(fr)   # pick up edits to the .py without restarting the kernel

pd.set_option('display.float_format', '{:,.0f}'.format)

## Parameters

In [2]:
REPORT_MONTH = None            # e.g. '2026-09'; None = latest month available
MONTHLY_BUDGET = 550_000       # AWS Services budget per month
INCLUDE_TAX_IN_BUDGET = False  # set True if Tax should count against the AWS budget

## Load & classify

In [3]:
df = fr.load_cost_data('.')
rd = fr.analyze(df, REPORT_MONTH, MONTHLY_BUDGET, INCLUDE_TAX_IN_BUDGET)
print(f"Months loaded: {df['Date'].nunique()}  ({df['Date'].min():%b %Y} – {df['Date'].max():%b %Y})")
print(f"Report month:  {rd.month:%B %Y}{'  ⚠ PARTIAL' if rd.m['partial'] else ''}")
rd.m['cat'].tail(13)

Months loaded: 60  (Oct 2021 – Sep 2026)
Report month:  September 2026


Category,AWS Services,Marketplace,Tax,Total,Budgeted
Date,,,,,
2025-09-01,"432,809","567,361","40,894","1,041,064","432,809"
2025-10-01,"415,784","149,531","15,743","581,058","415,784"
2025-11-01,"419,795",0,"5,870","425,665","419,795"
2025-12-01,"424,242","281,494","21,224","726,960","424,242"
2026-01-01,"434,117","486,505","33,977","954,600","434,117"
2026-02-01,"345,750",0,"11,233","356,983","345,750"
2026-03-01,"377,323","320,995","31,864","730,182","377,323"
2026-04-01,"384,616","48,062","13,280","445,958","384,616"
2026-05-01,"411,164","48,108","12,869","472,141","411,164"


## Executive summary

In [4]:
for title, key in [("What's working", 'wins'), ('Watch list', 'watch'), ('Recommended actions', 'actions')]:
    print(f'\n{title.upper()}')
    for line in rd.m[key]:
        print(f'  • {line}')


WHAT'S WORKING
  • AWS Services spend is $461K in September 2026, $202K/month (30%) below the peak 3-month average of $663K/month (ending Jun 2022).
  • Biggest structural reductions vs 2024 average: OpenSearch (-$51K/mo), EC2 Compute (-$40K/mo), S3 (-$27K/mo), RDS (-$26K/mo).
  • September 2026 is $89K under the $550K monthly AWS budget (84% utilized).
  • 2026 projected AWS Services spend at the current 3-month run-rate is $5.15M vs $6.60M annual budget ($1.45M headroom).

WATCH LIST
  • AWS Services has rebounded $116K/month (33%) since the Feb 2026 low of $346K. Main drivers: CloudFront (+$41K), CloudWatch (+$12K), RDS (+$8.5K), Enterprise Support (+$8.4K). Part of the low is artificial: no Savings Plans fee was billed that month.
  • CloudFront is growing: $59K/mo over the last 3 months vs $32K/mo the 3 months before.
  • Marketplace spend over the trailing 12 months is $2.27M ($189K/month on average), outside the AWS Services budget.

RECOMMENDED ACTIONS
  • Effective compute (E

## Journey & budget

In [5]:
fr.chart_journey(rd).show()
fr.chart_aws_trend(rd).show()
fr.chart_budget_ytd(rd).show()

In [6]:
rd.m['annual']

,Months,AWS Services,AWS avg / month,Marketplace,Tax,Total,AWS avg YoY,Mkt share
Date,,,,,,,,
2021,3,"1,279,066","426,355","3,729","11,236","1,294,031",NaN,0
2022,12,"7,041,239","586,770","1,495,787","140,269","8,677,296",0,0
2023,12,"6,956,643","579,720","3,451,824","273,105","10,681,572",-0,0
2024,12,"7,400,650","616,721","3,012,027","283,645","10,696,322",0,0
2025,12,"5,917,692","493,141","2,214,217","226,803","8,358,712",-0,0
2026,9,"3,773,333","419,259","1,835,995","200,126","5,809,455",-0,0


## AWS Services composition & movers

In [7]:
fr.chart_top_services(rd).show()
fr.chart_movers(rd, 'Δ vs prior 3 mo').show()
fr.chart_movers(rd, 'Δ vs peak yr').show()

In [8]:
rd.m['movers'].head(25)

,Current month,Last 3 mo avg,Prior 3 mo avg,Same 3 mo last yr,Peak-year avg,Trough month,Δ vs prior 3 mo,Δ vs last yr,Δ vs peak yr,Δ since trough,% vs last yr
Service,,,,,,,,,,,
Amazon CloudFront,"63,960","58,791","32,122","22,333","18,056","22,589","26,669","36,458","40,735","41,372",2
Amazon Relational Database Service,"54,520","56,128","50,944","60,471","81,748","46,052","5,184","-4,343","-25,620","8,467",-0
Savings Plans for AWS Compute usage,"53,653","55,503","41,528","39,744","51,861",0,"13,976","15,759","3,642","53,653",0
Amazon Elastic Compute Cloud - Compute,"57,736","54,576","49,881","69,485","94,788","71,654","4,695","-14,909","-40,213","-13,918",-0
AWS Support (Enterprise),"44,188","43,282","38,829","40,721","47,747","35,763","4,454","2,562","-4,465","8,426",0
Amazon Simple Storage Service,"43,954","41,984","40,598","42,811","68,989","43,774","1,387",-826,"-27,004",180,-0
EC2 - Other,"34,202","35,205","32,457","34,959","49,641","28,976","2,748",246,"-14,436","5,225",0
AmazonCloudWatch,"21,647","22,172","21,319","17,436","20,810","9,960",854,"4,736","1,362","11,687",0
Amazon ElastiCache,"11,165","13,529","13,263","17,078","25,611","3,631",266,"-3,548","-12,082","7,534",-0


## Commitments, Marketplace, tagging

In [9]:
fr.chart_commitments(rd).show()
fr.chart_marketplace_heat(rd).show()
fr.chart_tagging(rd).show()

In [10]:
display(rd.m['mkt_vendors'])
display(rd.m['renewals'])

,Trailing 12 mo,Prior 12 mo,Δ,Last charge
Vendor,,,,
Confluent (Kafka/Flink),"1,125,977","1,125,977",0,2026-09-01
CrowdStrike,"567,036","233,622","333,414",2026-07-01
Grafana Cloud,"240,367","285,867","-45,500",2026-09-01
Upwind,"192,000","144,000","48,000",2026-07-01
Tines,"48,000",0,"48,000",2026-05-01
NetSPI,"39,495","39,495",0,2026-03-01
Conduktor,"25,000",0,"25,000",2025-10-01
CloudFix,"22,500","22,500",0,2026-08-01
Anthropic Claude (Bedrock),"6,645",0,"6,645",2026-09-01


,Expected month,Vendor,Expected amount,Based on
0,2026-10-01,Conduktor,"25,000",2025-10-01
1,2026-10-01,CrowdStrike,"76,531",2025-10-01
2,2026-10-01,Upwind,"48,000",2025-10-01
3,2026-12-01,Confluent (Kafka/Flink),"281,494",2025-12-01
4,2027-01-01,CrowdStrike,"438,505",2026-01-01
5,2027-01-01,Upwind,"48,000",2026-01-01
6,2027-03-01,Confluent (Kafka/Flink),"281,494",2026-03-01
7,2027-03-01,NetSPI,"39,495",2026-03-01


## Write the HTML report

In [11]:
path = fr.build_report(rd, f'AWS_Executive_Report_{rd.month:%Y_%m}.html')
print('Report written:', path)

Report written: AWS_Executive_Report_2026_09.html


## (Optional) Pull a month with an authoritative Billing Entity
After running the SSO login cells in **FinOps.ipynb** (so `aws_profiles` exists), this pulls a month with the
`BillingEntity` column (AWS vs AWS Marketplace), which replaces the name-based classification:

```python
import finops_exec_report as fr
new = fr.fetch_month_costs(aws_profiles[0], 2026, 10)
new.to_csv('Monthly_Cost_For_OCT_2026.csv')
```